In [1]:
import json

In [22]:
with open('data/run_0__insurance_regulatory__gpt-oss-20b.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

In [23]:
data["data"] = data["data"][:1]

In [24]:
with open("data/5_insurance_regulatory.json", "w") as f:
    json.dump(data, f, indent=2)

In [2]:
%pip install datasets

Note: you may need to restart the kernel to use updated packages.


In [1]:
from datasets import load_dataset

ds = load_dataset("DKYoon/SlimPajama-6B")

/home/ubuntu/anjali/env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|██████████| 9346/9346 [00:00<00:00, 91474.79 examples/s]


In [7]:
ds['train'][0]

{'text': 'Want Tori to Coach You Too?\nTori\'s Health Step by Step coming soon.\nWin free copies, prizes, access to exclusive behind-the-scenes, free access to Coach Tori, and more.\nand receive a copy of Tori\'s Weekly Challenges. We\'ll also notify you of when Tori\'s Program becomes available.\nI\'ve been asked, even criticized, about adding a focus on nutrition to Desert. There\'s a reason why. I had poor nutritional examples growing up. Being confused on the issue of nutrition cost me a lot. I remember yo-yo\'ing a lot. The only time I even came close to being my desired weight was when I did high-intensity workouts daily. At one point, I was exercised about 6 hours a day. I was in multiple dance classes and a karate class, as well as another karate club that met for two hours three days a week. I also rode my bike to campus, and even added a one hour workout when I got home. I was still thirty pounds overweight. I can attest to the coined phrase "You cannot exercise away a bad di

In [ ]:
import json
from transformers import AutoTokenizer

S, O = 2048, 2048
MIN_TOKENS = 4096
TARGET_PER_FILE = 50_000

OUT_FILE = [
    "data/slimpajama_train_recon.json",
    "data/slimpajama_train_next.json",
]

tok = AutoTokenizer.from_pretrained("google/gemma-7b", use_fast=True)

def count_tokens(text: str) -> int:
    if not text:
        return 0

    return len(tok.encode(text, add_special_tokens=False, truncation=False))

writers = [open(p, "w", encoding="utf-8") for p in OUT_FILE]
counts = [0] * len(OUT_FILE)
file_idx = 0


try:
    for row in ds["train"]:
        if file_idx >= len(OUT_FILE):
            break

        article = row.get("text", "") or ""
        if count_tokens(article) < MIN_TOKENS:
            continue

        obj = {"tokens": article, "split": {"s": S, "o": O}}
        writers[file_idx].write(json.dumps(obj, ensure_ascii=False) + "\n")
        counts[file_idx] += 1

        if counts[file_idx] >= TARGET_PER_FILE:
            file_idx += 1

finally:
    for w in writers:
        w.close()

print("Done")

for p, c in zip(OUT_FILE, counts):
    print(f"{p}: {c}")

if sum(counts) < 3 * TARGET_PER_FILE:
    print(
        f"Warning: Insufficient data. Expected at least {3 * TARGET_PER_FILE} tokens, but got {sum(counts)}"
    )

        

In [4]:

with open('data/paragraph_train_next.json', 'r') as f:
    data = [json.loads(line) for line in f]

# print(data)

python refrag_new_impv.py cpt_recon   --train_json data/paragraph_train_recon.jsonl  --enc roberta-large   --dec google/gemma-7b   --k 16   --steps 5000   --lr 2e-5   --out_dir runs/cpt_recon_roblarge_gemma7b_16 --log_every 100

In [ ]:
instruction_prompt = """Answer this question only from the given context.
Question: 
Who is subject to the Shared Responsibility Payment in New Jersey?
Context:

Answer:
      """

In [13]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-9b")
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2-9b",
    torch_dtype=torch.bfloat16,   # Use bfloat16 for memory efficiency
    device_map="auto"              # Automatically use GPU if available
)

def generate_answer(prompt: str, max_new_tokens: int = 512) -> str:
    """Generate an answer using Gemma-2-9B."""
    
    # Tokenize the input
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Generate output
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.2,
            top_p=0.9,
            top_k=50,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode only the newly generated tokens (exclude input prompt)
    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    
    return answer.strip()


Loading checkpoint shards: 100%|██████████| 8/8 [00:01<00:00,  5.04it/s]


In [14]:
print("Answer:", generate_answer(instruction_prompt, max_new_tokens=256))

Answer: Subject to the shared responsibility payment in new jersey
